# Figure 4 — Source Data Export

**Figure 4** shows the hierarchical alignment between primate visual areas and
GAN priors: per-area success rates (V1/V4/IT), population-level activation
trajectories normalised to the maximum response, and per-experiment time
constants of the evolution dynamics.

## Data requirements

> **Raw neural recordings are required to run this notebook.**
> The `.mat`/`.pkl` files are not distributed with the repository.
> Set the `MATROOT` environment variable to the directory containing
> `Both_BigGAN_FC6_Evol_Stats.pkl` and `Both_BigGAN_FC6_Evol_Stats_expsep.pkl`
> before launching Jupyter, or assign the path directly to `matroot` in the
> cell below.
>
> Pre-computed time-constant fit results are expected in the `tabdir` path.
>
> ```bash
> export MATROOT=/path/to/Mat_Statistics
> ```

The exported source data files are written to `../source_data/Fig4/`.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join
from scipy.stats import sem

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))
from neuro_data_analysis.neural_data_lib import (
    load_neural_data,
    extract_all_evol_trajectory_psth,
    pad_psth_traj,
    get_expstr,
    extract_natref_activation_array,
    extract_evol_activation_array,
)
from neuro_data_analysis.neural_data_utils import get_all_masks

In [ ]:
# Set the path to the raw neural recording data.
# Override by setting the MATROOT environment variable before launching Jupyter.
matroot = os.environ.get("MATROOT", "")

BFEStats_merge, BFEStats = load_neural_data()

# Build experiment-level metadata and standard inclusion masks
from neuro_data_analysis.neural_data_lib import extract_all_evol_trajectory_dyna, pad_resp_traj
resp_col, meta_df = extract_all_evol_trajectory_dyna(BFEStats)
Amsk, Bmsk, V1msk, V4msk, ITmsk, length_msk, spc_msk, sucsmsk, \
    bsl_unstable_msk, bsl_stable_msk, validmsk = get_all_masks(meta_df)

# Pad trajectories to a common length
resp_extrap_arr, extrap_mask_arr, max_len = pad_resp_traj(resp_col)

tabdir = join(matroot, "Tables")
outdir = "../source_data/Fig4"
os.makedirs(outdir, exist_ok=True)

print(f"Valid experiments: {validmsk.sum()}")

## Figure 4A: Success rates

Compute the fraction of successfully evolving experiments for each visual area
(V1, V4, IT) separately for DeePSim (thread 0) and BigGAN (thread 1).
Success is defined as a significant t-test (p < 0.05) between the max-block
response and the initial-block response.

In [ ]:
area_masks = {"V1": V1msk, "V4": V4msk, "IT": ITmsk}
sucs0_msk = meta_df.p_maxinit_0 < 0.05   # DeePSim success
sucs1_msk = meta_df.p_maxinit_1 < 0.05   # BigGAN success

records = []
for area, amsk in area_masks.items():
    base = validmsk & amsk
    n_total = base.sum()
    records.append({
        "area": area,
        "n_total": int(n_total),
        "DeePSim_sucs": int((base & sucs0_msk).sum()),
        "BigGAN_sucs":  int((base & sucs1_msk).sum()),
        "DeePSim_rate": (base & sucs0_msk).sum() / n_total if n_total > 0 else np.nan,
        "BigGAN_rate":  (base & sucs1_msk).sum() / n_total if n_total > 0 else np.nan,
    })

sucs_rate_df = pd.DataFrame(records)
sucs_rate_df.to_csv(join(outdir, "Fig4A_SuccessRate_maxinit.csv"), index=False)

# Plain-text label string for axis annotations
label_lines = [f"{r['area']}: DeePSim {r['DeePSim_rate']:.2f}, BigGAN {r['BigGAN_rate']:.2f}" for r in records]
with open(join(outdir, "Fig4A_sucs_labels.txt"), "w") as f:
    f.write("\n".join(label_lines) + "\n")

print(sucs_rate_df)
print("Figure 4A source data saved.")

## Figure 4B/C: Activation trajectories

Export population-mean and SEM activation trajectories, normalised so the
maximum response across generations equals 1.  Separate CSVs are produced
for DeePSim and BigGAN threads.

In [ ]:
# resp_extrap_arr: (n_exps, max_gen, 6)
# columns: mean0, mean1, sem0, sem1, bsl_mean0, bsl_mean1
# extrap_mask_arr: (n_exps, max_gen) — True where padded (no data)

valid_idx = np.where(validmsk.values)[0]
resp_valid = resp_extrap_arr[valid_idx]          # (N, max_gen, 6)
mask_valid = extrap_mask_arr[valid_idx]           # (N, max_gen)

# Normalise each experiment by its own maximum response (threads 0 and 1)
max_resp = np.nanmax(np.where(~mask_valid[:, :, np.newaxis], resp_valid[:, :, :2], np.nan), axis=1, keepdims=True)
resp_norm = resp_valid[:, :, :2] / (max_resp + 1e-8)   # (N, max_gen, 2)
sem_norm  = resp_valid[:, :, 2:4] / (max_resp + 1e-8)  # (N, max_gen, 2)

# Mask padded entries as NaN before computing population stats
resp_norm[mask_valid] = np.nan
sem_norm[mask_valid]  = np.nan

pop_mean = np.nanmean(resp_norm, axis=0)   # (max_gen, 2)
pop_sem  = np.nanmean(sem_norm,  axis=0)   # (max_gen, 2)

for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
    pd.DataFrame(pop_mean[:, thr_idx]).to_csv(
        join(outdir, f"Figure4_src_maxnorm_resp_stats_all_exps_{space_name}_mean_data.csv"), index=False)
    pd.DataFrame(pop_sem[:, thr_idx]).to_csv(
        join(outdir, f"Figure4_src_maxnorm_resp_stats_all_exps_{space_name}_sem_data.csv"), index=False)

# Export the mask and labels dataframe
mask_df = meta_df[validmsk].copy()
mask_df["sucs_DeePSim"] = sucs0_msk[validmsk].values
mask_df["sucs_BigGAN"]  = sucs1_msk[validmsk].values
mask_df.to_csv(join(outdir, "Figure4_src_exp_masks_and_succ_labels.csv"), index=False)

# Export extrapolation mask
pd.DataFrame(extrap_mask_arr[valid_idx]).to_csv(
    join(outdir, "Figure4_src_extrapolation_mask_arr.csv"), index=False)

print("Figure 4B/C source data saved.")

## Figure 4D: Time constants

Load pre-computed exponential fit time constants and merge with experiment
metadata (visual area, animal, success label) before exporting.

In [ ]:
# Pre-computed time constant fits are produced by:
#   analysis/evol_time_constant_fitting.py
tau_df = pd.read_csv(join(tabdir, "evol_traj_time_constant_w_meta.csv"))

# Merge with master metadata to ensure area / success labels are current
tau_merged = tau_df.merge(
    meta_df[["Expi", "visual_area", "Animal", "p_maxinit_0", "p_maxinit_1"]],
    on="Expi", how="left", suffixes=("_fit", "_meta")
)

tau_merged.to_csv(
    join(outdir, "Figure4_src_evol_traj_time_constant_w_meta.csv"), index=False
)

print(f"Time constant table shape: {tau_merged.shape}")
print("Figure 4D source data saved to", outdir)